# Task 4: Advanced Analytics (Basic)

**ApexPlanet Software Pvt. Ltd. — Data Analytics Internship**

**Objective:** Statistical analysis, time series, clustering, and predictive modeling.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

df = pd.read_csv("../data/ecommerce_sales_cleaned.csv")
df["Order_Date"] = pd.to_datetime(df["Order_Date"])
print("✅ All libraries loaded!")
print("Dataset shape:", df.shape)

## Day 21-22: Statistical Analysis

In [1]:
# Descriptive Statistics
print("DESCRIPTIVE STATISTICS")
print("=" * 60)
desc = df[["Net_Sales", "Profit", "Quantity", "Discount_Percent"]].agg(["mean","median","std","skew"])
print(desc.round(2))

In [1]:
# T-Test: Express vs Standard shipping order values
express = df[df["Ship_Mode"] == "Express"]["Net_Sales"]
standard = df[df["Ship_Mode"] == "Standard"]["Net_Sales"]

t_stat, p_val = stats.ttest_ind(express, standard, equal_var=False)

print("T-TEST: Express vs Standard Shipping")
print("=" * 45)
print(f"Express  mean: ₹{express.mean():,.2f}  (n={len(express)})")
print(f"Standard mean: ₹{standard.mean():,.2f}  (n={len(standard)})")
print(f"t-statistic  : {t_stat:.3f}")
print(f"p-value      : {p_val:.4f}")
print(f"Result: {'Significant difference ✅' if p_val < 0.05 else 'No significant difference ❌'}")

In [1]:
# Chi-Square Test: Region vs Category
contingency = pd.crosstab(df["Region"], df["Category"])
chi2, p_val, dof, expected = stats.chi2_contingency(contingency)

print("CHI-SQUARE TEST: Region vs Category")
print("=" * 45)
print(f"Chi2 = {chi2:.3f}, dof = {dof}, p-value = {p_val:.4f}")
print(f"Result: {'Significant association ✅' if p_val < 0.05 else 'No significant association ❌'}")

In [1]:
# 95% Confidence Interval
mean_s = df["Net_Sales"].mean()
ci = stats.t.interval(0.95, len(df)-1, loc=mean_s, scale=stats.sem(df["Net_Sales"]))
print(f"95% Confidence Interval for mean Net Sales:")
print(f"Mean = ₹{mean_s:,.2f}")
print(f"95% CI = (₹{ci[0]:,.2f}, ₹{ci[1]:,.2f})")

## Day 23-24: Time Series Analysis

In [1]:
# Monthly time series
ts = df.groupby(df["Order_Date"].dt.to_period("M"))["Net_Sales"].sum()
ts.index = ts.index.to_timestamp()
trend = ts.rolling(window=3, center=True, min_periods=1).mean()
residual = ts - trend

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
ts.plot(ax=axes[0], color="#4C72B0", marker="o")
axes[0].set_title("Observed Monthly Net Sales")
trend.plot(ax=axes[1], color="#DD8452", marker="o")
axes[1].set_title("Trend (3-month rolling mean)")
residual.plot(ax=axes[2], color="#55A868", marker="o")
axes[2].axhline(0, color="grey", linestyle="--", linewidth=0.8)
axes[2].set_title("Residual")
plt.tight_layout()
plt.savefig("../reports/chart_timeseries.png", dpi=150)
plt.show()
print("✅ Time series decomposition done!")

In [1]:
# Moving Average Forecast
ma3 = ts.rolling(window=3).mean()
forecast_idx = pd.date_range(ts.index[-1] + pd.offsets.MonthBegin(1), periods=3, freq="MS")
forecast = pd.Series([ma3.iloc[-1]]*3, index=forecast_idx)

fig, ax = plt.subplots(figsize=(11, 5))
ts.plot(ax=ax, label="Actual", marker="o", color="#4C72B0")
ma3.plot(ax=ax, label="3-Month Moving Avg", color="#C44E52", linestyle="--")
forecast.plot(ax=ax, label="Forecast (next 3 months)", color="#55A868", marker="s", linestyle=":")
ax.set_title("Monthly Net Sales — Moving Average Forecast")
ax.legend()
plt.tight_layout()
plt.show()
print("Forecast for next 3 months:")
print(forecast.round(0))

## Customer Segmentation (K-Means Clustering)

In [1]:
# Prepare customer features
cust = df.groupby("Customer_ID").agg(
    total_spend=("Net_Sales", "sum"),
    avg_order_value=("Net_Sales", "mean"),
    order_count=("Order_ID", "count"),
    avg_discount=("Discount_Percent", "mean"),
).reset_index()

X = cust[["total_spend", "avg_order_value", "order_count", "avg_discount"]]
X_scaled = StandardScaler().fit_transform(X)
print("Customer features shape:", cust.shape)
cust.describe().round(2)

In [1]:
# Elbow method to find best k
inertias = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(2, 9), inertias, marker="o", color="#8172B2", linewidth=2)
ax.set_xlabel("Number of Clusters (k)")
ax.set_ylabel("Inertia")
ax.set_title("Elbow Method — Finding Optimal k")
plt.tight_layout()
plt.show()
print("✅ Based on the elbow, k=4 is optimal")

In [1]:
# Apply K-Means with k=4
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cust["cluster"] = kmeans.fit_predict(X_scaled)

# PCA for 2D visualization
pca_result = PCA(n_components=2).fit_transform(X_scaled)
cust["pca1"] = pca_result[:, 0]
cust["pca2"] = pca_result[:, 1]

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(cust["pca1"], cust["pca2"],
                     c=cust["cluster"], cmap="viridis", alpha=0.6, s=30)
ax.set_xlabel("PCA Component 1")
ax.set_ylabel("PCA Component 2")
ax.set_title("Customer Segments (K=4) — PCA View")
ax.legend(*scatter.legend_elements(), title="Cluster")
plt.tight_layout()
plt.savefig("../reports/chart_clusters.png", dpi=150)
plt.show()
print("✅ Clustering done!")

In [1]:
# Cluster profiles
profile = cust.groupby("cluster").agg(
    customers=("Customer_ID","count"),
    avg_spend=("total_spend","mean"),
    avg_orders=("order_count","mean"),
    avg_discount=("avg_discount","mean"),
).round(2)
print("CLUSTER PROFILES:")
print(profile)

## Day 25-26: Basic Predictive Model

In [1]:
# Linear Regression to predict Net_Sales
features = pd.get_dummies(
    df[["Quantity","Unit_Price","Discount_Percent","Category","Region","Customer_Segment","Payment_Method","Ship_Mode"]],
    drop_first=True
)
target = df["Net_Sales"]

X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("LINEAR REGRESSION — Predict Net Sales")
print("=" * 40)
print(f"R²   = {r2:.4f}")
print(f"MAE  = ₹{mae:,.2f}")
print(f"RMSE = ₹{rmse:,.2f}")

# Top 3 features
coefs = pd.Series(model.coef_, index=features.columns).sort_values(key=abs, ascending=False)
print("\nTop 3 most important features:")
print(coefs.head(3))

In [1]:
# Actual vs Predicted plot
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, y_pred, alpha=0.3, color="#4C72B0", s=15)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, color="red", linestyle="--", linewidth=1.5)
ax.set_xlabel("Actual Net Sales (₹)")
ax.set_ylabel("Predicted Net Sales (₹)")
ax.set_title(f"Actual vs Predicted (R² = {r2:.3f})")
plt.tight_layout()
plt.savefig("../reports/chart_model.png", dpi=150)
plt.show()
print("\n✅ Task 4 Complete!")